# Transformer + frozen GloVe

Runs the whole experiment in two rounds, dev only (test is never touched here):

1. **Round 1:** the default and the 7 ablations in `src/sc4002/experiments.py`, seed 42 each. Each
   ablation changes one setting from the default.
2. **Round 2:** the 3 best round-1 ablations and the default, with seeds 42, 43 and 44. The final
   configuration is the one with the highest **mean** dev combined score.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/SC4002_WL'
RES = f'{WORK}/results'
os.makedirs(f'{WORK}/embeddings', exist_ok=True)
os.makedirs(RES, exist_ok=True)
os.environ['SC4002_EMB_DIR'] = f'{WORK}/embeddings'
os.environ['SC4002_RESULTS'] = RES
os.environ['SC4002_CKPT'] = f'{WORK}/checkpoints'

Mounted at /content/drive


## Get the code
**Option A** (after WL_branch is pushed): clone it.
**Option B**: upload `wl_code.zip` (made locally) to `MyDrive/SC4002_WL/` and unzip it.

In [ ]:
CODE = '/content/SC4002'
USE_GIT = False 

if USE_GIT:
    !rm -rf {CODE} && git clone -q -b WL_branch https://github.com/masamune-prog/SC4002.git {CODE}
else:
    !rm -rf {CODE} && mkdir -p {CODE} && unzip -q -o {WORK}/wl_code.zip -d {CODE}
!pip install -q tokenizers pytest
os.environ['PYTHONPATH'] = f'{CODE}/src'
import sys; sys.path.insert(0, f'{CODE}/src')
%cd {CODE}

/content/SC4002


## GloVe 6B word vectors (downloaded once to Drive, ~860 MB)

In [ ]:
E = os.environ['SC4002_EMB_DIR']
if not os.path.exists(f'{E}/glove.6B.zip'):
    !wget -q -O {E}/glove.6B.zip https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
!ls -la {E}

total 1155057
drwx------ 2 root root      4096 Oct 28 15:22 .
drwx------ 4 root root      4096 Oct 29 05:43 ..
-rw------- 1 root root 862182613 Oct 25  2015 glove.6B.zip
-rw------- 1 root root 160293346 Oct 28 16:19 glove_store_1c9bd92a47_100000.npz
-rw------- 1 root root 160293346 Oct 28 15:26 glove_store_ba6f54df91_100000.npz


## Unit tests

In [ ]:
!python -m pytest -q tests

................                                                         [100%]
16 passed in 6.82s


## Round 1: default + 7 ablations, seed 42

In [ ]:
from sc4002.experiments import CORE
!python -m sc4002.train --exp {' '.join(CORE)} --seeds 42

=== default seed 42 on cuda
units 5304  chars 34  frozen table 38190x300  params total 19,999,735  trainable 8,542,735
epoch  1  train loss 3.8344  dev loss 1.4352  dev intent acc 70.93  slot F1 54.92  combined 62.92  18.7s  *
epoch  2  train loss 1.9108  dev loss 0.9737  dev intent acc 81.70  slot F1 68.77  combined 75.24  15.6s  *
epoch  3  train loss 1.6249  dev loss 0.8964  dev intent acc 82.83  slot F1 72.79  combined 77.81  15.1s  *
epoch  4  train loss 1.4982  dev loss 0.8574  dev intent acc 83.96  slot F1 72.86  combined 78.41  15.5s  *
epoch  5  train loss 1.4039  dev loss 0.8020  dev intent acc 84.60  slot F1 74.28  combined 79.44  15.2s  *
epoch  6  train loss 1.3381  dev loss 0.7744  dev intent acc 85.74  slot F1 75.86  combined 80.80  15.4s  *
epoch  7  train loss 1.2820  dev loss 0.7988  dev intent acc 84.85  slot F1 76.35  combined 80.60  15.2s
epoch  8  train loss 1.2308  dev loss 0.8171  dev intent acc 85.19  slot F1 76.51  combined 80.85  15.1s  *
epoch  9  train loss

In [ ]:
import json, pandas as pd

def load_runs():
    rows = [json.loads(l) for l in open(f'{RES}/ablations.jsonl') if l.strip()]
    return pd.DataFrame([{'exp': r['exp'], 'seed': r['config']['seed'], 'best_epoch': r['best_epoch'],
                          'intent': r['dev_intent_acc'], 'slot': r['dev_slot_f1'], 'combined': r['dev_combined'],
                          'total_params': r['total_params'], 'trainable_params': r['trainable_params'],
                          'minutes': r['train_time'] / 60} for r in rows])

r1 = load_runs().query('seed == 42').set_index('exp')
missing = [e for e in CORE if e not in r1.index]
assert not missing, f'round 1 not finished, missing: {missing}'
r1['delta_vs_default'] = r1['combined'] - r1.loc['default', 'combined']
r1.loc[CORE].sort_values('combined', ascending=False).round(2)

,seed,best_epoch,intent,slot,combined,total_params,trainable_params,minutes,delta_vs_default
exp,,,,,,,,,
no_char,42,31,87.85,82.34,85.10,19999747,8493547,9.90,0.31
tok_clitic,42,32,87.70,82.33,85.01,19999735,8523535,10.09,0.23
default,42,37,87.65,81.91,84.78,19999735,8542735,10.31,0.00
no_intent_cond,42,29,87.31,81.40,84.36,19999819,8514319,10.15,-0.43
tok_bpe2k,42,33,87.16,81.31,84.24,19999831,8106031,10.52,-0.54
dec_softmax,42,32,88.19,78.65,83.42,19999792,8530192,7.09,-1.36
topk_0,42,30,86.52,80.17,83.35,10033135,8542735,10.20,-1.44
no_glove,42,39,86.08,80.46,83.27,8427535,8427535,9.26,-1.51


## Round 2: top-3 ablations and the default, seeds 42, 43, 44

In [ ]:
TOP3 = r1.drop(index='default').sort_values('combined', ascending=False).index[:3].tolist()
CANDIDATES = ['default'] + TOP3
SEEDS = [42, 43, 44]
print('Round 2 candidates:', CANDIDATES)
!python -m sc4002.train --exp {' '.join(CANDIDATES)} --seeds {' '.join(map(str, SEEDS))}

Round 2 candidates: ['default', 'no_char', 'tok_clitic', 'no_intent_cond']
=== default seed 42: already done, skipping
=== default seed 43 on cuda
units 5304  chars 34  frozen table 38190x300  params total 19,999,735  trainable 8,542,735
epoch  1  train loss 3.8219  dev loss 1.4273  dev intent acc 72.21  slot F1 56.65  combined 64.43  17.2s  *
epoch  2  train loss 1.9213  dev loss 1.0528  dev intent acc 79.59  slot F1 69.57  combined 74.58  16.2s  *
epoch  3  train loss 1.6342  dev loss 0.9335  dev intent acc 81.51  slot F1 71.13  combined 76.32  16.2s  *
epoch  4  train loss 1.5016  dev loss 0.8835  dev intent acc 83.87  slot F1 73.83  combined 78.85  15.7s  *
epoch  5  train loss 1.4050  dev loss 0.8830  dev intent acc 83.03  slot F1 75.87  combined 79.45  15.8s  *
epoch  6  train loss 1.3367  dev loss 0.8396  dev intent acc 84.90  slot F1 75.62  combined 80.26  15.6s  *
epoch  7  train loss 1.2809  dev loss 0.7957  dev intent acc 85.44  slot F1 77.48  combined 81.46  15.8s  *
epoch 

## 3-seed comparison and final choice
Selection rule: highest mean dev combined = (intent acc + slot F1) / 2 over the three seeds.

In [ ]:
df = load_runs()
df = df[df.exp.isin(CANDIDATES) & df.seed.isin(SEEDS)]
missing = [(e, s) for e in CANDIDATES for s in SEEDS if not ((df.exp == e) & (df.seed == s)).any()]
assert not missing, f'runs still missing: {missing}'
print(df.sort_values(['exp', 'seed']).round(2).to_string(index=False))
summary = df.groupby('exp').agg(intent=('intent', 'mean'), slot=('slot', 'mean'),
                                combined=('combined', 'mean'), combined_std=('combined', 'std'),
                                min_combined=('combined', 'min'), total_params=('total_params', 'first'),
                                trainable_params=('trainable_params', 'first'))
summary['delta_vs_default'] = summary['combined'] - summary.loc['default', 'combined']
summary = summary.sort_values('combined', ascending=False)
summary.round(2)

           exp  seed  best_epoch  intent  slot  combined  total_params  trainable_params  minutes
       default    42          37   87.65 81.91     84.78      19999735           8542735    10.31
       default    43          37   87.46 81.94     84.70      19999735           8542735    10.62
       default    44          32   87.80 82.23     85.02      19999735           8542735    10.91
       no_char    42          31   87.85 82.34     85.10      19999747           8493547     9.90
       no_char    43          26   87.06 82.13     84.60      19999747           8493547    10.29
       no_char    44          37   86.92 82.12     84.52      19999747           8493547    10.32
no_intent_cond    42          29   87.31 81.40     84.36      19999819           8514319    10.15
no_intent_cond    43          32   88.10 81.55     84.82      19999819           8514319    10.36
no_intent_cond    44          40   87.85 82.10     84.98      19999819           8514319    10.34
    tok_clitic    42

,intent,slot,combined,combined_std,min_combined,total_params,trainable_params,delta_vs_default
exp,,,,,,,,
default,87.64,82.03,84.83,0.17,84.70,19999735,8542735,0.00
no_char,87.28,82.20,84.74,0.31,84.52,19999747,8493547,-0.10
no_intent_cond,87.75,81.69,84.72,0.32,84.36,19999819,8514319,-0.11
tok_clitic,87.51,81.62,84.56,0.50,84.02,19999735,8523535,-0.27


In [ ]:
from sc4002.experiments import EXPERIMENTS
best = summary.index[0]
d = summary.loc[best, 'delta_vs_default']
noise = summary['combined_std'].max()
print(f'Final configuration: {best}  (mean dev combined {summary.loc[best, "combined"]:.2f}, '
      f'{d:+.2f} vs default; largest seed std {noise:.2f})')
if best != 'default' and d < noise:
    print('Note: the margin over the default is smaller than the seed spread, so the difference is not clearly real.')
print('Overrides vs default:', EXPERIMENTS[best])

Final configuration: default  (mean dev combined 84.83, +0.00 vs default; largest seed std 0.50)
Overrides vs default: {}
